corregir_alcaldias.py

Corrige asignaciones de alcaldía en el dataset de accidentes CDMX

mediante spatial join (vecino más cercano) con GeoPandas.

In [2]:
import geopandas as gpd
import pandas as pd
import unicodedata
from pathlib import Path

# ---------------------------------------------------------------
# 0. Configuración de rutas (ajustadas a la nueva estructura)
# ---------------------------------------------------------------
BASE_DIR = Path(r"D:\Datasets\2026_accidentesCDMX")

RUTA_FACT = BASE_DIR / "Data" / "Processed" / "fact_accidentes2.csv"
RUTA_SHP = BASE_DIR / "Data" / "External" / "alcaldias" / "alcaldias.shp"
RUTA_SALIDA = BASE_DIR / "Data" / "Processed" / "fact_accidentes_corregido.csv"

UMBRAL_SOSPECHOSO_M = 500  # metros; ajustable según tu criterio

In [4]:
# ---------------------------------------------------------------
# 1. Función auxiliar de normalización de texto
# ---------------------------------------------------------------
def normalizar_texto(texto):
    """Mayúsculas, sin acentos, sin espacios extra — para comparar
    'CUAUHTEMOC' (csv) contra 'Cuauhtémoc' (shapefile) sin falsos positivos."""
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto

In [5]:
# ---------------------------------------------------------------
# 2. Cargar tabla de hechos y construir geometría de puntos
# ---------------------------------------------------------------
def cargar_accidentes(ruta_csv):
    df = pd.read_csv(ruta_csv)

    columnas_requeridas = {"longitud", "latitud", "alcaldia"}
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas en el CSV: {faltantes}")

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitud, df.latitud),
        crs="EPSG:4326",
    )
    return gdf


# ---------------------------------------------------------------
# 3. Cargar shapefile de alcaldías y normalizar nombres
# ---------------------------------------------------------------
def cargar_alcaldias(ruta_shp):
    alcaldias = gpd.read_file(ruta_shp).to_crs("EPSG:4326")

    if "NOMGEO" not in alcaldias.columns:
        raise ValueError(
            "El shapefile no tiene columna 'NOMGEO'. "
            f"Columnas disponibles: {list(alcaldias.columns)}"
        )

    alcaldias["NOMGEO_norm"] = alcaldias["NOMGEO"].apply(normalizar_texto)
    return alcaldias


# ---------------------------------------------------------------
# 4. Spatial join por vecino más cercano
# ---------------------------------------------------------------
def hacer_spatial_join(gdf, alcaldias):
    # Reproyectar a CRS métrico para medir distancias reales en metros
    gdf_m = gdf.to_crs("EPSG:6372")
    alc_m = alcaldias.to_crs("EPSG:6372")

    resultado = gpd.sjoin_nearest(
        gdf_m,
        alc_m,
        how="left",
        distance_col="dist_a_alcaldia_m",
    )
    return resultado


# ---------------------------------------------------------------
# 5. Comparar alcaldía original vs calculada, marcar sospechosos
# ---------------------------------------------------------------
def validar_y_marcar(resultado, umbral_m):
    resultado["alcaldia_original_norm"] = resultado["alcaldia"].apply(normalizar_texto)
    resultado["alcaldia_corregida"] = resultado["NOMGEO_norm"]
    resultado["discrepancia"] = (
        resultado["alcaldia_original_norm"] != resultado["alcaldia_corregida"]
    )
    resultado["coordenada_sospechosa"] = resultado["dist_a_alcaldia_m"] > umbral_m
    return resultado


# ---------------------------------------------------------------
# 6. Pipeline principal
# ---------------------------------------------------------------
def main():
    print(f"Cargando accidentes desde: {RUTA_FACT}")
    gdf = cargar_accidentes(RUTA_FACT)

    print(f"Cargando shapefile de alcaldías desde: {RUTA_SHP}")
    alcaldias = cargar_alcaldias(RUTA_SHP)

    print("Ejecutando spatial join (nearest)...")
    resultado = hacer_spatial_join(gdf, alcaldias)

    print("Validando alcaldía original vs corregida...")
    resultado = validar_y_marcar(resultado, UMBRAL_SOSPECHOSO_M)

    # Quitar geometría antes de exportar (Tableau no la necesita)
    salida = resultado.drop(columns="geometry")

    RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
    salida.to_csv(RUTA_SALIDA, index=False)

    # --- Resumen ---
    total = len(salida)
    n_discrepancia = int(salida["discrepancia"].sum())
    n_sospechoso = int(salida["coordenada_sospechosa"].sum())

    print("\n--- Resumen ---")
    print(f"Total de registros procesados: {total}")
    print(f"Discrepancias de alcaldía (declarada != calculada): {n_discrepancia} "
          f"({n_discrepancia/total:.1%})")
    print(f"Coordenadas sospechosas (>{UMBRAL_SOSPECHOSO_M} m de cualquier alcaldía): "
          f"{n_sospechoso} ({n_sospechoso/total:.1%})")
    print(f"Archivo corregido guardado en: {RUTA_SALIDA}")

    if n_discrepancia > 0:
        print("\nMuestra de discrepancias (primeras 10):")
        print(
            salida.loc[
                salida["discrepancia"],
                ["id_accidente", "alcaldia", "alcaldia_corregida", "dist_a_alcaldia_m"],
            ].head(10).to_string(index=False)
        )


if __name__ == "__main__":
    main()

Cargando accidentes desde: D:\Datasets\2026_accidentesCDMX\Data\Processed\fact_accidentes2.csv
Cargando shapefile de alcaldías desde: D:\Datasets\2026_accidentesCDMX\Data\External\alcaldias\alcaldias.shp


DataSourceError: D:\Datasets\2026_accidentesCDMX\Data\External\alcaldias\alcaldias.shp: No such file or directory

In [8]:
from pathlib import Path

carpeta = Path(r"D:\Datasets\2026_accidentesCDMX\Data\External\alcaldias")

print("¿Existe la carpeta?", carpeta.exists())

if carpeta.exists():
    for archivo in carpeta.iterdir():
        print(archivo.name)
else:
    print("La carpeta no existe todavía.")

¿Existe la carpeta? True


In [11]:
import geopandas as gpd
from pathlib import Path

ruta_shp = Path(r"D:\Datasets\2026_accidentesCDMX\Data\External\alcaldias\poligonos_alcaldias_cdmx")
alcaldias = gpd.read_file(ruta_shp)

print(alcaldias.columns.tolist())
print(alcaldias.head(30))

['CVEGEO', 'CVE_ENT', 'CVE_MUN', 'NOMGEO', 'geometry']
   CVEGEO CVE_ENT CVE_MUN                  NOMGEO  \
0   09002      09     002            Azcapotzalco   
1   09003      09     003                Coyoacán   
2   09004      09     004   Cuajimalpa de Morelos   
3   09005      09     005       Gustavo A. Madero   
4   09006      09     006               Iztacalco   
5   09007      09     007              Iztapalapa   
6   09008      09     008  La Magdalena Contreras   
7   09009      09     009              Milpa Alta   
8   09010      09     010          Álvaro Obregón   
9   09011      09     011                 Tláhuac   
10  09012      09     012                 Tlalpan   
11  09013      09     013              Xochimilco   
12  09014      09     014           Benito Juárez   
13  09015      09     015              Cuauhtémoc   
14  09016      09     016          Miguel Hidalgo   
15  09017      09     017     Venustiano Carranza   

                                           

In [14]:
"""
corregir_alcaldias.py
Corrige asignaciones de alcaldía en el dataset de accidentes CDMX
mediante spatial join (vecino más cercano) con GeoPandas.

Ubicación sugerida: Accidentes-CDMX/scripts/corregir_alcaldias.py
"""
import geopandas as gpd
import pandas as pd
import unicodedata
from pathlib import Path

# ---------------------------------------------------------------
# 0. Configuración de rutas
# ---------------------------------------------------------------
BASE_DIR = Path(r"D:\Datasets\2026_accidentesCDMX")

RUTA_FACT = BASE_DIR / "Data" / "Processed" / "fact_accidentes2.csv"
RUTA_SHP = BASE_DIR / "Data" / "External" / "alcaldias" / "poligonos_alcaldias_cdmx"
RUTA_SALIDA = BASE_DIR / "Data" / "Processed" / "fact_accidentes_corregido.csv"

CAMPO_ALCALDIA_SHP = "NOMGEO"   # confirmado en el shapefile de datos.cdmx.gob.mx
UMBRAL_SOSPECHOSO_M = 500       # metros; ajustable según tu criterio


# ---------------------------------------------------------------
# 1. Función auxiliar de normalización de texto
# ---------------------------------------------------------------
def normalizar_texto(texto):
    """Mayúsculas, sin acentos, sin espacios extra — para comparar
    'CUAUHTEMOC' (csv) contra 'Cuauhtémoc' (shapefile) sin falsos positivos."""
    if pd.isna(texto):
        return texto
    texto = str(texto).upper().strip()
    texto = unicodedata.normalize("NFKD", texto).encode("ascii", "ignore").decode("utf-8")
    return texto


# ---------------------------------------------------------------
# 2. Cargar tabla de hechos y construir geometría de puntos
# ---------------------------------------------------------------
def cargar_accidentes(ruta_csv):
    df = pd.read_csv(ruta_csv)

    columnas_requeridas = {"longitud", "latitud", "alcaldia"}
    faltantes = columnas_requeridas - set(df.columns)
    if faltantes:
        raise ValueError(f"Faltan columnas esperadas en el CSV: {faltantes}")

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df.longitud, df.latitud),
        crs="EPSG:4326",
    )
    return gdf


# ---------------------------------------------------------------
# 3. Cargar shapefile de alcaldías y normalizar nombres
# ---------------------------------------------------------------
def cargar_alcaldias(ruta_shp):
    alcaldias = gpd.read_file(ruta_shp).to_crs("EPSG:4326")

    if CAMPO_ALCALDIA_SHP not in alcaldias.columns:
        raise ValueError(
            f"El shapefile no tiene columna '{CAMPO_ALCALDIA_SHP}'. "
            f"Columnas disponibles: {list(alcaldias.columns)}"
        )

    alcaldias["NOMGEO_norm"] = alcaldias[CAMPO_ALCALDIA_SHP].apply(normalizar_texto)
    return alcaldias


# ---------------------------------------------------------------
# 4. Spatial join por vecino más cercano
# ---------------------------------------------------------------
def hacer_spatial_join(gdf, alcaldias):
    # Reproyectar a CRS métrico para medir distancias reales en metros
    gdf_m = gdf.to_crs("EPSG:6372")
    alc_m = alcaldias.to_crs("EPSG:6372")

    resultado = gpd.sjoin_nearest(
        gdf_m,
        alc_m,
        how="left",
        distance_col="dist_a_alcaldia_m",
    )
    return resultado

# ---------------------------------------------------------------
# 5. Comparar alcaldía original vs calculada, marcar sospechosos
# ---------------------------------------------------------------
def validar_y_marcar(resultado, umbral_m):
    resultado["alcaldia_original_norm"] = resultado["alcaldia"].apply(normalizar_texto)
    resultado["alcaldia_corregida"] = resultado["NOMGEO_norm"]
    resultado["discrepancia"] = (
        resultado["alcaldia_original_norm"] != resultado["alcaldia_corregida"]
    )
    resultado["coordenada_sospechosa"] = resultado["dist_a_alcaldia_m"] > umbral_m
    return resultado

# ---------------------------------------------------------------
# 6. Pipeline principal
# ---------------------------------------------------------------
def main():
    print(f"Cargando accidentes desde: {RUTA_FACT}")
    gdf = cargar_accidentes(RUTA_FACT)

    print(f"Cargando shapefile de alcaldías desde: {RUTA_SHP}")
    alcaldias = cargar_alcaldias(RUTA_SHP)

    print("Ejecutando spatial join (nearest)...")
    resultado = hacer_spatial_join(gdf, alcaldias)

    print("Validando alcaldía original vs corregida...")
    resultado = validar_y_marcar(resultado, UMBRAL_SOSPECHOSO_M)

    # Quitar geometría antes de exportar (Tableau no la necesita)
    salida = resultado.drop(columns="geometry")

    RUTA_SALIDA.parent.mkdir(parents=True, exist_ok=True)
    salida.to_csv(RUTA_SALIDA, index=False)

    # --- Resumen ---
    total = len(salida)
    n_discrepancia = int(salida["discrepancia"].sum())
    n_sospechoso = int(salida["coordenada_sospechosa"].sum())

    print("\n--- Resumen ---")
    print(f"Total de registros procesados: {total}")
    print(f"Discrepancias de alcaldía (declarada != calculada): {n_discrepancia} "
          f"({n_discrepancia/total:.1%})")
    print(f"Coordenadas sospechosas (>{UMBRAL_SOSPECHOSO_M} m de cualquier alcaldía): "
          f"{n_sospechoso} ({n_sospechoso/total:.1%})")
    print(f"Archivo corregido guardado en: {RUTA_SALIDA}")

    if n_discrepancia > 0:
        print("\nMuestra de discrepancias (primeras 10):")
        print(
            salida.loc[
                salida["discrepancia"],
                ["id_accidente", "alcaldia", "alcaldia_corregida", "dist_a_alcaldia_m"],
            ].head(1000).to_string(index=False)
        )


if __name__ == "__main__":
    main()

Cargando accidentes desde: D:\Datasets\2026_accidentesCDMX\Data\Processed\fact_accidentes2.csv
Cargando shapefile de alcaldías desde: D:\Datasets\2026_accidentesCDMX\Data\External\alcaldias\poligonos_alcaldias_cdmx
Ejecutando spatial join (nearest)...
Validando alcaldía original vs corregida...

--- Resumen ---
Total de registros procesados: 4657
Discrepancias de alcaldía (declarada != calculada): 682 (14.6%)
Coordenadas sospechosas (>500 m de cualquier alcaldía): 0 (0.0%)
Archivo corregido guardado en: D:\Datasets\2026_accidentesCDMX\Data\Processed\fact_accidentes_corregido.csv

Muestra de discrepancias (primeras 10):
 id_accidente            alcaldia     alcaldia_corregida  dist_a_alcaldia_m
            5 VENUSTIANO CARRANZA              IZTACALCO           0.000000
            6    GUSTAVO A MADERO      GUSTAVO A. MADERO           0.000000
            9    GUSTAVO A MADERO      GUSTAVO A. MADERO           0.000000
           21    GUSTAVO A MADERO      GUSTAVO A. MADERO           0.